# 06 — Dollar Bars & Residual Mean-Reversion Re-test

Notebook 05 showed that on 1-minute wall-clock bars the PC1-stripped residual MR signal has gross Sharpe ~2.4 but per-trade edge of only ~0.2 bps, versus a ~1.3 bps breakeven at 2 bps round-trip cost — a ~6× gap. The suspicion is that 1-minute bars are the wrong clock: they mix high-information and zero-information intervals.

**This notebook does two things:**
1. Builds **dollar bars** from 7 days of BTC aggTrades (each bar = a fixed traded notional, e.g. $10M).
2. Re-tests the residual MR signal on those bars and compares per-trade edge vs the 1-min baseline.

Because we only have ~7 days of aggTrades (BTC), the sample is small — this is a structural sanity check, not a definitive backtest.

In [1]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

REPO = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

from src.backtest import run_backtest

pd.set_option('display.float_format', lambda x: f'{x:,.4f}')
plt.rcParams['figure.figsize'] = (11, 4)

AGG_DIR = REPO / 'dataset' / 'binance' / 'aggTrades' / 'BTCUSDT'

## 1. Load 7 days of BTC aggTrades

In [2]:
files = sorted(AGG_DIR.glob('BTCUSDT-aggTrades-*.parquet'))
print(f'{len(files)} files:')
for f in files:
    print(' ', f.name)

trades = pd.concat([pd.read_parquet(f) for f in files], ignore_index=True)
trades = trades.sort_values('timestamp').reset_index(drop=True)

# Signed quantity: is_buyer_maker=True  -> the buyer was passive (maker), so the
# aggressor was the seller -> seller-initiated, signed -1.
# is_buyer_maker=False -> the seller was passive, aggressor is the buyer -> +1.
trades['signed_qty'] = np.where(trades['is_buyer_maker'], -trades['qty'], trades['qty'])
trades['notional']   = trades['price'] * trades['qty']

print(f'\n{len(trades):,} trades from {trades.timestamp.min()} to {trades.timestamp.max()}')
print(f'Total notional traded: ${trades.notional.sum()/1e9:.2f}B')
trades.head()

7 files:
  BTCUSDT-aggTrades-2026-04-01.parquet
  BTCUSDT-aggTrades-2026-04-02.parquet
  BTCUSDT-aggTrades-2026-04-03.parquet
  BTCUSDT-aggTrades-2026-04-04.parquet
  BTCUSDT-aggTrades-2026-04-05.parquet
  BTCUSDT-aggTrades-2026-04-06.parquet
  BTCUSDT-aggTrades-2026-04-07.parquet

9,145,930 trades from 2026-04-01 00:00:00.049000+00:00 to 2026-04-07 23:59:59.962000+00:00
Total notional traded: $66.88B


,agg_id,price,qty,first_id,last_id,timestamp,is_buyer_maker,signed_qty,notional
0,3231966237,"68,241.4000",1.2840,7513550890,7513550893,2026-04-01 00:00:00.049000+00:00,True,-1.2840,"87,621.9576"
1,3231966238,"68,241.5000",0.0490,7513550894,7513550896,2026-04-01 00:00:00.095000+00:00,False,0.0490,"3,343.8335"
2,3231966256,"68,238.4000",0.0040,7513550938,7513550938,2026-04-01 00:00:00.111000+00:00,True,-0.0040,272.9536
3,3231966255,"68,238.5000",0.0030,7513550936,7513550937,2026-04-01 00:00:00.111000+00:00,True,-0.0030,204.7155
4,3231966254,"68,238.8000",0.0060,7513550934,7513550935,2026-04-01 00:00:00.111000+00:00,True,-0.0060,409.4328


## 2. Build dollar bars

Each bar closes once cumulative notional since the previous close crosses a threshold. We target roughly the same bar count as 1-minute bars over the same 7-day window (~10k bars), which for ~$50B of notional means a bar every **$5M**.

In [3]:
BAR_NOTIONAL = 5_000_000  # $5M per bar

def build_dollar_bars(tr: pd.DataFrame, threshold: float) -> pd.DataFrame:
    """Aggregate trades into bars, each containing exactly ~`threshold` dollars of notional."""
    price = tr['price'].to_numpy()
    qty   = tr['qty'].to_numpy()
    nt    = tr['notional'].to_numpy()
    sq    = tr['signed_qty'].to_numpy()
    ts    = tr['timestamp'].to_numpy()

    bars = []
    cum_not = 0.0
    start = 0
    for i in range(len(tr)):
        cum_not += nt[i]
        if cum_not >= threshold:
            sl = slice(start, i + 1)
            p_sl = price[sl]
            q_sl = qty[sl]
            bars.append({
                'close_time':   ts[i],
                'open':         p_sl[0],
                'high':         p_sl.max(),
                'low':          p_sl.min(),
                'close':        p_sl[-1],
                'volume':       q_sl.sum(),
                'notional':     nt[sl].sum(),
                'trade_count':  i + 1 - start,
                'signed_vol':   sq[sl].sum(),
                'vwap':         float((p_sl * q_sl).sum() / q_sl.sum()),
            })
            start = i + 1
            cum_not = 0.0
    return pd.DataFrame(bars).set_index('close_time')

bars = build_dollar_bars(trades, BAR_NOTIONAL)
bars['returns']     = bars['close'].pct_change()
bars['log_returns'] = np.log(bars['close'] / bars['close'].shift(1))

span_hours = (bars.index[-1] - bars.index[0]) / np.timedelta64(1, 'h')
print(f'{len(bars):,} dollar bars over {span_hours:.1f} hours')
print(f'Average bars per hour: {len(bars)/span_hours:.1f}')
print(f'Bar notional target:   ${BAR_NOTIONAL:,}')
print(f'Actual mean notional:  ${bars["notional"].mean():,.0f}')
bars.head()

12,798 dollar bars over 168.0 hours
Average bars per hour: 76.2
Bar notional target:   $5,000,000
Actual mean notional:  $5,225,468


,open,high,low,close,volume,notional,trade_count,signed_vol,vwap,returns,log_returns
close_time,,,,,,,,,,,
2026-04-01 00:00:41.744000+00:00,"68,241.4000","68,241.5000","68,194.1000","68,194.1000",74.0670,"5,052,342.5539",1036,-36.4750,"68,213.1388",NaN,NaN
2026-04-01 00:01:06.590000+00:00,"68,194.1000","68,202.2000","68,139.0000","68,139.7000",73.8710,"5,036,280.3501",811,-42.7070,"68,176.6911",-0.0008,-0.0008
2026-04-01 00:01:06.690000+00:00,"68,139.8000","68,154.5000","68,121.8000","68,128.7000",73.6190,"5,015,914.8628",402,-52.7170,"68,133.4284",-0.0002,-0.0002
2026-04-01 00:01:11.775000+00:00,"68,128.2000","68,130.5000","68,088.2000","68,101.3000",73.7250,"5,021,592.8754",598,-59.5090,"68,112.4839",-0.0004,-0.0004
2026-04-01 00:01:29.102000+00:00,"68,094.8000","68,101.2000","68,072.5000","68,094.3000",73.4740,"5,002,676.7415",896,6.6680,"68,087.7146",-0.0001,-0.0001


## 3. Signal-quality comparison: 1-min bars vs dollar bars

Quick diagnostic: how does the lag-1 autocorrelation of returns compare between the two bar types? More negative autocorr = stronger mean-reversion per bar. We don't have ETH/SOL aggTrades, so we can't do the full PCA residual on dollar bars — instead we use the BTC return itself as a proxy for 'the thing the MR signal wants to trade'.

In [4]:
# 1-minute wall-clock bars for the same window, for comparison.
tr_idx = trades.set_index('timestamp')
minute = pd.DataFrame({
    'close':    tr_idx['price'].resample('1min').last(),
    'volume':   tr_idx['qty'].resample('1min').sum(),
    'notional': tr_idx['notional'].resample('1min').sum(),
}).dropna()
minute['returns']     = minute['close'].pct_change()
minute['log_returns'] = np.log(minute['close'] / minute['close'].shift(1))

def lag1(x): return float(x.dropna().autocorr(lag=1))
def stats(r, name):
    r = r.dropna()
    return pd.Series({
        'n':        len(r),
        'mean':     r.mean(),
        'std':      r.std(),
        'abs_mean': r.abs().mean(),
        'lag1_ac':  lag1(r),
    }, name=name)

comp = pd.concat([
    stats(minute['log_returns'],  '1-min bars'),
    stats(bars['log_returns'],    'dollar bars'),
], axis=1)
comp

,1-min bars,dollar bars
n,"10,079.0000","12,797.0000"
mean,0.0000,0.0000
std,0.0005,0.0005
abs_mean,0.0003,0.0004
lag1_ac,0.0349,0.0012


## 4. Residual MR signal on dollar bars

We can't strip PC1 here (no synchronous ETH/SOL ticks), so we use a pure BTC mean-reversion z-score on the dollar-bar returns and compare per-trade edge vs the minute-bar version. The question is whether dollar-bar timing alone sharpens the edge enough to survive costs.

In [5]:
def mr_zscore_signal(bar_df: pd.DataFrame, lookback: int = 60, smooth_span: int = 5) -> pd.Series:
    r = bar_df['log_returns']
    mu = r.rolling(lookback).mean()
    sd = r.rolling(lookback).std()
    z  = -((r - mu) / sd)
    z  = z.clip(-3, 3).ewm(span=smooth_span).mean().fillna(0)
    return (z / 3.0)

def annualized_sharpe_bar(pnl: pd.Series, bars_per_year: float) -> float:
    r = pnl.dropna()
    if len(r) == 0 or r.std() == 0: return float('nan')
    return float(r.mean() / r.std() * np.sqrt(bars_per_year))

def backtest_grid(bar_df: pd.DataFrame, bars_per_year: float, label: str) -> pd.DataFrame:
    sig = mr_zscore_signal(bar_df, lookback=60, smooth_span=5)
    # run_backtest wants a 'returns' column; make sure it's there.
    if 'returns' not in bar_df.columns:
        bar_df = bar_df.assign(returns=bar_df['log_returns'])
    rows = []
    for cost in [0.0, 0.5, 2.0]:
        res = run_backtest(bar_df, sig, cost_bps=cost)
        turnover_sum = float(res.turnover.sum())
        gross_pnl    = float(res.pnl.sum() + cost/10_000 * turnover_sum)  # add back applied cost
        rows.append({
            'bars':      label,
            'cost_bps':  cost,
            'sharpe':    annualized_sharpe_bar(res.pnl, bars_per_year),
            'total_ret': float(res.pnl.sum()),
            'max_dd':    float((res.cumulative_pnl - res.cumulative_pnl.cummax()).min()),
            'turnover':  turnover_sum,
            'n_bars':    len(bar_df),
            # Per-round-trip gross edge in bps.
            'edge_bps':  gross_pnl / max(turnover_sum, 1) * 10_000,
        })
    return pd.DataFrame(rows)

# Bars-per-year: 1-min bars = 525600. Dollar bars projected from sample rate.
minute_bpy = 525_600
dollar_bpy = len(bars) / span_hours * 24 * 365

g_min = backtest_grid(minute, minute_bpy, '1-min')
g_dol = backtest_grid(bars,   dollar_bpy, 'dollar')

pd.concat([g_min, g_dol], ignore_index=True)

,bars,cost_bps,sharpe,total_ret,max_dd,turnover,n_bars,edge_bps
0,1-min,0.0000,22.4595,0.0286,-0.0061,904.9083,10080,0.3158
1,1-min,0.5000,-13.0913,-0.0167,-0.0202,904.9083,10080,0.3158
2,1-min,2.0000,-118.2875,-0.1524,-0.1530,904.9083,10080,0.3158
3,dollar,0.0000,-1.9560,-0.0023,-0.0082,"1,199.9674",12798,-0.0188
4,dollar,0.5000,-53.8760,-0.0623,-0.0626,"1,199.9674",12798,-0.0188
5,dollar,2.0000,-205.8586,-0.2423,-0.2423,"1,199.9674",12798,-0.0188


## 5. Read the comparison

Two numbers to look at:

- **`edge_bps`** — the per-round-trip gross edge in basis points. This is the cost floor the strategy needs to clear. A good MR signal on dollar bars should push this from ~0.2 bps (1-min baseline) toward the 1–2 bps range.
- **Gross Sharpe ratio** — annualized using the bar-type-specific bars-per-year, so the two numbers are directly comparable.

If `edge_bps` improves meaningfully on dollar bars, the bar-type confound is real and the next step is to (a) run this on a longer aggTrades sample, and (b) rebuild the cross-asset residual on a synchronized dollar-bar clock. If it doesn't improve, the MR signal itself is weak and we pivot the pitch to a negative-result narrative with a different next experiment (e.g. condition on funding-rate divergence).

---
### Caveats
- 7-day sample — purely a directional check, not a conclusion.
- Pure BTC MR here, not the cross-asset PC1-residual version from notebook 05.
- Dollar bars give irregular time spacing; the `bars_per_year` Sharpe annualization is approximate.